# Grid Cells From a Network That Was Never Asked

**A representation nobody asked for is still an answer to a question somebody asked** · GPU · ~50 min · Colab

A rat in the dark still knows where it is. It feels its own speed and turning and keeps a running sum: path integration. In 2005, recordings from the rat's entorhinal cortex found neurons that fire at the corners of a triangular lattice laid across the whole room. They were named grid cells, and nothing in the rat's world is hexagonal. Here you hand a small recurrent network the same problem: velocity in, "where am I?" out. Hexagons are never mentioned. Then you open the network and draw where each hidden unit fires. You also train a twin that differs in one detail only, to find out what the hexagons were really a response to.

### The goal

Both networks learn to track their own position from velocity alone, and the centre-surround network does it measurably less precisely. Even so, it turns a clear share of its units into stable hexagonal firing maps, while its better-navigating Gaussian twin produces almost none.

### The papers behind this

- [recurrent-neural-network](https://azimuth.plus/en/paper/recurrent-neural-network) — a hidden state that carries the position forward from step to step
- [backpropagation-through-time](https://azimuth.plus/en/paper/backpropagation-through-time) — how an error at the last step reaches the weights used at the first

> Save a copy to Drive before you start (File → Save a copy in Drive). Edits to the original are not saved.

## Setup

`PROFILE` is the only scale knob. The free tier is the default and stays inside Colab's free envelope.

In [ ]:
SLUG = "emergent-grid-cells"
LANG = "en"
PROFILE = "free"  # free | a100

# Colab defaults to inline figures; CI does not. Being explicit means the
# captured plot on the site and the plot you see are produced the same way.
%matplotlib inline

# The shim lives in this repository, not on PyPI: a workshop should never
# depend on a package index staying up.
import os
import subprocess
import sys
from pathlib import Path

# Guarded three ways. Colab users re-run the setup cell constantly, and
# a contributor may already be sitting inside a checkout — the first
# Windows run of this notebook cloned the repository into its own
# generated/notebooks/ directory because neither case was handled.
# The hash of the code.py THESE CELLS were built from. setup()
# compares it with the code.py it finds on disk: if a notebook is
# older than its source, every number below describes code that is
# not the code anyone is reading. The printed `code ·` line was
# taken from disk and so could not catch this by itself.
os.environ['AZIMUTH_NOTEBOOK_CODEHASH'] = 'd210db7a8469ff8c'

REPO = 'azimuth-workshops'
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'shim' / 'azimuth_nb').is_dir()), None)
if root is None:
    if not Path(REPO).exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', 'stable',
             'https://github.com/MotazSabri/azimuth-workshops.git', REPO],
            check=True,
        )
    root = (here / REPO).resolve()
print('workshop root:', root)

# Absolute, so nothing depends on the working directory. Dropping the
# %cd this cell used to do also means the notebook stops caring where
# it was opened from.
sys.path.insert(0, str(root / 'shim'))
_ = os.environ.setdefault('AZIMUTH_DATA_DIR', str(root / 'data'))

Close your eyes and walk three steps forward, turn left, walk two more. You still have a fair idea where the door is. Nothing told you; you integrated your own motion. Animals do this constantly, and in the entorhinal cortex of the rat some of the neurons involved have a startling signature: plotted against position, each one fires at the vertices of a triangular lattice that tiles the entire floor.

This workshop asks whether an artificial network, given only the same problem, arrives at the same answer — and then asks the harder question of what exactly made it do so.

_Confirm the profile and the code hash; everything below is sized from that profile._

In [ ]:
import azimuth_nb as azimuth

env = azimuth.setup(SLUG, lang=LANG, profile=PROFILE)

The network's output is not a pair of coordinates. It predicts the activity of {{scale.place_cells}} simulated place cells scattered over a {{scale.box_m}}-metre box, each most active near its own centre. The workshop trains two versions that differ in the shape of that tuning. In one, each place cell is a plain Gaussian bump. In the other, the bump is ringed by a shallow zone of suppression — a centre-surround profile. Walks, starting weights, network and training schedule are identical. Hold on to that single difference; the rest of the workshop turns on it.

_Left: six simulated walks over grey place-cell centres; when a walk reaches a wall it slows and slides along it. Right: one place cell's activity along a line through its centre, orange for centre-surround and blue for Gaussian. The orange curve never goes negative. It sits on a raised floor and is notched down to zero in a ring around the peak, and that ring is the surround. Because the code must stay non-negative and sum to one, about three quarters of the centre-surround code is that nearly flat floor, which is why this network's loss will barely move._

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np
import torch

assert torch.__version__, "torch comes with the runtime; it is never installed here"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = env.cfg
BOX = cfg["box_m"]
DT = 0.02  # seconds per step
torch.manual_seed(cfg["seed"])

# One fixed set of place-cell centres, shared by both networks.
centers = (
    torch.rand(cfg["place_cells"], 2, generator=torch.Generator().manual_seed(cfg["seed"])) - 0.5
) * BOX
centers = centers.to(device)


def make_trajectories(rng, batch, steps, box=BOX):
    """Smooth random walks that turn away from walls. Returns positions (batch, steps+1, 2)."""
    sigma_turn = 11.52  # rad/s, rotational velocity spread
    speed_scale = 0.13 * 2 * math.pi  # m/s, Rayleigh scale of forward speed
    border = 0.03
    pos = np.zeros((batch, steps + 1, 2))
    pos[:, 0] = rng.uniform(-box / 2, box / 2, (batch, 2))
    heading = rng.uniform(0, 2 * math.pi, batch)
    turns = rng.normal(0, sigma_turn, (batch, steps))
    speeds = rng.rayleigh(speed_scale, (batch, steps))
    for t in range(steps):
        x, y = pos[:, t, 0], pos[:, t, 1]
        dists = np.stack([box / 2 - x, box / 2 - y, box / 2 + x, box / 2 + y])
        wall_angle = dists.argmin(0) * math.pi / 2
        toward = np.mod(heading - wall_angle + math.pi, 2 * math.pi) - math.pi
        near = (dists.min(0) < border) & (np.abs(toward) < math.pi / 2)
        v = np.where(near, 0.25, 1.0) * speeds[:, t]
        heading = (
            heading
            + np.where(near, np.sign(toward) * (math.pi / 2 - np.abs(toward)), 0.0)
            + DT * turns[:, t]
        )
        pos[:, t + 1] = pos[:, t] + (v * DT)[:, None] * np.stack(
            [np.cos(heading), np.sin(heading)], -1
        )
    return pos


def place_code(pos, surround):
    """Population activity of the place cells at positions (..., 2). Sums to 1 over cells.

    surround=None gives plain Gaussian bumps. surround=s subtracts a wider bump
    (variance scaled by s): a centre that excites, ringed by a zone that inhibits.
    """
    d2 = ((pos[..., None, :] - centers) ** 2).sum(-1)
    width2 = cfg["place_sigma_m"] ** 2
    out = torch.softmax(-d2 / (2 * width2), -1)
    if surround is not None:
        out = out - torch.softmax(-d2 / (2 * surround * width2), -1)
        out = out - out.min(-1, keepdim=True).values
        out = out / out.sum(-1, keepdim=True)
    return out


# Picture the task: a few walks, and one place cell's tuning under each target.
rng = np.random.default_rng(cfg["seed"])
walks = make_trajectories(rng, 6, 200)
fig, (ax_walk, ax_tune) = plt.subplots(1, 2, figsize=(9, 4))
c = centers.cpu().numpy()
ax_walk.scatter(c[:, 0], c[:, 1], s=4, color="0.8")
for w in walks:
    ax_walk.plot(w[:, 0], w[:, 1], lw=1)
ax_walk.set_xlim(-BOX / 2, BOX / 2)
ax_walk.set_ylim(-BOX / 2, BOX / 2)
ax_walk.set_aspect("equal")

nearest = int((centers**2).sum(-1).argmin())  # the place cell closest to the middle
xs = torch.linspace(-BOX / 2, BOX / 2, 400, device=device)
line = torch.stack([xs, torch.full_like(xs, centers[nearest, 1].item())], -1)
for surround, colour in [(cfg["surround_scale"], "tab:orange"), (None, "tab:blue")]:
    tuning = place_code(line, surround)[:, nearest].cpu().numpy()
    ax_tune.plot(xs.cpu().numpy(), tuning / tuning.max(), color=colour, lw=2)
ax_tune.axhline(0, color="0.6", lw=0.8)
ax_tune.set_ylim(-0.3, 1.1)
plt.tight_layout()
plt.show()

if env.lang == "ar":
    hardware = "بطاقة رسوميات" if device.type == "cuda" else "المعالج المركزي"
    print(f"الساحة {BOX} م × {BOX} م · {cfg['place_cells']} خلية مكان · التشغيل على {hardware}")
else:
    print(f"arena {BOX} m × {BOX} m · {cfg['place_cells']} place cells · device: {device}")

The hidden state starts from the place-cell code of the starting point. From then on the network receives one thing per step: how far it moved in x and y. To predict the place code {{scale.seq_len}} steps on, it must accumulate those movements inside its hidden state. The loss compares predicted and true place-cell activity. It contains no term about spatial structure, periodicity, or angles.

_An encoder for the starting place, a ReLU recurrent layer, and a linear readout — nothing more._

In [ ]:
from torch import nn


class PathIntegrator(nn.Module):
    """Velocity in, place-cell prediction out. The hidden layer is never told what to be."""

    def __init__(self, n_place, n_hidden):
        super().__init__()
        self.encoder = nn.Linear(
            n_place, n_hidden, bias=False
        )  # starting place -> first hidden state
        self.rnn = nn.RNN(2, n_hidden, nonlinearity="relu", bias=False, batch_first=True)
        self.decoder = nn.Linear(n_hidden, n_place, bias=False)

    def hidden(self, velocity, start_code):
        states, _ = self.rnn(velocity, self.encoder(start_code)[None])
        return states

    def forward(self, velocity, start_code):
        return self.decoder(self.hidden(velocity, start_code))


def decode(logits):
    """Position estimate: the mean centre of the three most active predicted place cells."""
    return centers[logits.topk(3, dim=-1).indices].mean(-2)


def batch_tensors(pos, surround):
    pos = torch.as_tensor(pos, dtype=torch.float32, device=device)
    velocity = pos[:, 1:] - pos[:, :-1]
    return pos, velocity, place_code(pos[:, 0], surround), place_code(pos[:, 1:], surround)


params = sum(
    p.numel() for p in PathIntegrator(cfg["place_cells"], cfg["hidden_units"]).parameters()
)
if env.lang == "ar":
    print(f"{cfg['hidden_units']} وحدة مخفية · {params / 1e6:.1f} مليون معامل")
else:
    print(f"{cfg['hidden_units']} hidden units · {params / 1e6:.1f}M parameters")

> **On scale** — Each network has {{scale.hidden_units}} hidden units and trains for {{scale.train_steps}} steps on batches of {{scale.batch}} walks of {{scale.seq_len}} steps. Training the two networks is the long part of this workshop, and the comparison needs both. Published models of this kind train considerably longer; expect fainter lattices than in the literature, not different ones.

_Watch the position error decoded from the predicted place cells, not the loss, which barely moves. Expect a long plateau. In the reference run the error sat near a metre for about eleven thousand steps, broke between roughly twelve and twenty thousand, and ended near 9 cm. Do not stop the cell while the error is flat: position error is the whole of what the network is rewarded for, and the break comes late._

In [ ]:
import time


def train(surround):
    torch.manual_seed(cfg["seed"])  # identical initial weights for both networks
    rng = np.random.default_rng(cfg["seed"])  # identical trajectories for both networks
    model = PathIntegrator(cfg["place_cells"], cfg["hidden_units"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["learning_rate"])
    # Half precision for the recurrent arithmetic on a GPU. A T4 in full precision
    # ran 6.4 steps/s here, almost all of it matrix multiplication. The loss stays
    # in full precision, and the scaler keeps its very small gradients from
    # rounding to zero.
    amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    history, started = [], time.time()
    for step in range(1, cfg["train_steps"] + 1):
        pos, velocity, start, target = batch_tensors(
            make_trajectories(rng, cfg["batch"], cfg["seq_len"]), surround
        )
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
            logits = model(velocity, start)
        logits = logits.float()
        loss = -(target * torch.log_softmax(logits, -1)).sum(-1).mean()
        loss = loss + cfg["weight_decay"] * (model.rnn.weight_hh_l0**2).sum()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if step % cfg["log_every"] == 0 or step == cfg["train_steps"]:
            with torch.no_grad():
                err_cm = 100 * (decode(logits) - pos[:, 1:]).norm(dim=-1).mean().item()
            history.append((step, loss.item(), err_cm))
            rate = step / (time.time() - started)
            if env.lang == "ar":
                print(
                    f"خطوة {step:>6} · الخسارة {loss.item():.3f} · خطأ الموضع {err_cm:.1f} سم · {rate:.1f} خطوة/ث"
                )
            else:
                print(
                    f"step {step:>6} · loss {loss.item():.3f} · position error {err_cm:.1f} cm · {rate:.1f} steps/s"
                )
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    return model.eval(), np.array(history), time.time() - started, peak


dog_model, dog_history, dog_seconds, dog_peak_gb = train(cfg["surround_scale"])
train_minutes_dog = round(dog_seconds / 60, 1)
peak_vram_gb = round(dog_peak_gb, 2)

_The same walks in the same order, from the same starting weights. In the plot, the horizontal axis is training steps and the vertical one is position error in centimetres. The blue curve drops within a few thousand steps; the orange one gets there much later and settles higher. If the blue one never comes down, the twin did not learn the task and nothing later about it means anything._

In [ ]:
control_model, control_history, control_seconds, _ = train(None)
train_minutes_control = round(control_seconds / 60, 1)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(dog_history[:, 0], dog_history[:, 2], color="tab:orange", lw=2)
ax.plot(control_history[:, 0], control_history[:, 2], color="tab:blue", lw=2)
ax.set_ylim(bottom=0)  # zero-based: both curves must visibly reach the floor
plt.tight_layout()
plt.show()

To see what a hidden unit represents, walk the trained network through thousands of fresh trajectories and average that unit's activity in each square of a {{scale.map_res}}×{{scale.map_res}} floor grid. The result is its firing map. To score it, correlate the map with shifted copies of itself, which turns any repeating pattern into a ring of peaks around the centre. Rotate that picture. A hexagonal lattice matches itself at 60° and 120° and clashes at 30°, 90° and 150°; the grid score is the gap between the two. Squares and stripes score near or below zero.

_Skill compares each network with a guess that never leaves the starting point: 1 is perfect, 0 learned nothing. Both must be well above zero before any map below is worth reading. They will not be equal. The Gaussian twin navigates more precisely, {{run.metrics.control_skill}} against {{run.metrics.dog_skill}} in the reference run, and that gap matters for everything that follows._

In [ ]:
@torch.no_grad()
def survey(model, surround, seq_len, skip=0):
    """Walk the trained network through fresh trajectories.

    Returns (rate maps [units, res, res], stability [units], decoding skill,
    error in cm, error in cm of always guessing the box centre).
    Only steps at index >= skip are binned and scored. Skill = 1 - model error /
    error of a guess that never moves from the starting point, so 0 means "did
    not integrate". That guess gets worse as walks get longer, so skill is only
    comparable between walks of the same length; the centimetre errors are not.
    Stability correlates the maps built from two halves of the walks: a real map
    agrees with itself, noise does not.
    """
    res = cfg["map_res"]
    rng = np.random.default_rng(cfg["seed"] + 1)  # unseen trajectories, same for both networks
    sums = torch.zeros(2, res * res, cfg["hidden_units"], device=device)
    counts = torch.zeros(2, res * res, device=device)
    model_err, still_err, centre_err = 0.0, 0.0, 0.0
    for batch in range(cfg["eval_batches"]):
        pos, velocity, start, _ = batch_tensors(
            make_trajectories(rng, cfg["batch"], seq_len), surround
        )
        states = model.hidden(velocity, start)[:, skip:]
        where = pos[:, 1 + skip :]
        model_err += (decode(model.decoder(states)) - where).norm(dim=-1).mean().item()
        still_err += (pos[:, :1] - where).norm(dim=-1).mean().item()
        centre_err += where.norm(dim=-1).mean().item()
        cell = ((where + BOX / 2) / BOX * res).long().clamp(0, res - 1)
        index = (cell[..., 0] * res + cell[..., 1]).reshape(-1)
        half = batch % 2
        sums[half].index_add_(0, index, states.reshape(-1, states.shape[-1]))
        counts[half].index_add_(0, index, torch.ones_like(index, dtype=torch.float32))
    maps = (sums.sum(0) / counts.sum(0).clamp(min=1)[:, None]).T.reshape(-1, res, res).cpu().numpy()
    halves = sums / counts.clamp(min=1)[..., None]  # (2, bins, units)
    seen = (counts > 0).all(0)
    a, b = halves[0][seen], halves[1][seen]
    a, b = a - a.mean(0), b - b.mean(0)
    stability = (
        ((a * b).sum(0) / ((a * a).sum(0) * (b * b).sum(0)).sqrt().clamp(min=1e-12)).cpu().numpy()
    )
    per_batch_cm = 100 / cfg["eval_batches"]
    return (
        maps,
        stability,
        1 - model_err / still_err,
        model_err * per_batch_cm,
        centre_err * per_batch_cm,
    )


dog_maps, dog_stability, dog_skill, dog_err_cm, _ = survey(
    dog_model, cfg["surround_scale"], cfg["seq_len"]
)
control_maps, control_stability, control_skill, control_err_cm, _ = survey(
    control_model, None, cfg["seq_len"]
)
dog_skill, control_skill = round(dog_skill, 3), round(control_skill, 3)

if env.lang == "ar":
    print(
        f"مهارة تكامل المسار · هدف المركز والمحيط {dog_skill:.3f} · الهدف الغاوسي {control_skill:.3f}"
    )
    print(
        f"خطأ الموضع · هدف المركز والمحيط {dog_err_cm:.1f} سم · الهدف الغاوسي {control_err_cm:.1f} سم"
    )
else:
    print(
        f"path-integration skill · centre-surround {dog_skill:.3f} · Gaussian {control_skill:.3f}"
    )
    print(
        f"position error · centre-surround {dog_err_cm:.1f} cm · Gaussian {control_err_cm:.1f} cm"
    )

_The {{scale.units_shown}} highest-scoring hidden units of the centre-surround network, each drawn over the floor of the box, with its grid score above. Look for bumps arranged in triangles — every unit with its own spacing, orientation and offset. At {{scale.map_res}}×{{scale.map_res}} bins the maps are coarse, and a lattice this wide fits only a few periods in the box, so some units show just three or four bumps. The printed line underneath matters as much as the picture. Before training, the same network draws speckled maps, and speckle can score above 1 by pure chance. So a grid unit must also draw the same map from two separate halves of the walks, which a lattice does and speckle never does._

In [ ]:
def autocorrelogram(maps):
    """Normalised spatial autocorrelation of each map, shape (units, 2*res-1, 2*res-1)."""
    n, res, _ = maps.shape
    size = 2 * res - 1
    x = np.zeros((n, size, size))
    x[:, :res, :res] = maps - maps.mean(axis=(1, 2), keepdims=True)
    ones = np.zeros((1, size, size))
    ones[:, :res, :res] = 1.0

    def xcorr(a, b):
        return np.fft.fftshift(
            np.real(np.fft.ifft2(np.fft.fft2(a) * np.conj(np.fft.fft2(b)))), axes=(1, 2)
        )

    overlap = np.round(xcorr(ones, ones))
    s_a, s_b = xcorr(x, ones), xcorr(ones, x)
    s_ab, s_aa, s_bb = xcorr(x, x), xcorr(x**2, ones), xcorr(ones, x**2)
    num = overlap * s_ab - s_a * s_b
    den = np.sqrt(
        np.clip(overlap * s_aa - s_a**2, 0, None) * np.clip(overlap * s_bb - s_b**2, 0, None)
    )
    sac = np.where((den > 1e-12) & (overlap >= 20), num / np.maximum(den, 1e-12), 0.0)
    return sac  # zero lag sits at index res-1 on both axes


def rotate(images, degrees):
    """Bilinear rotation about the centre, keeping the frame."""
    _, h, w = images.shape
    theta = math.radians(degrees)
    yy, xx = np.mgrid[0:h, 0:w].astype(float)
    cy, cx = (h - 1) / 2, (w - 1) / 2
    src_x = math.cos(theta) * (xx - cx) + math.sin(theta) * (yy - cy) + cx
    src_y = -math.sin(theta) * (xx - cx) + math.cos(theta) * (yy - cy) + cy
    x0, y0 = np.floor(src_x).astype(int), np.floor(src_y).astype(int)
    fx, fy = src_x - x0, src_y - y0
    out = np.zeros_like(images)
    for dy, dx, weight in [
        (0, 0, (1 - fx) * (1 - fy)),
        (0, 1, fx * (1 - fy)),
        (1, 0, (1 - fx) * fy),
        (1, 1, fx * fy),
    ]:
        yi, xi = y0 + dy, x0 + dx
        inside = (yi >= 0) & (yi < h) & (xi >= 0) & (xi < w)
        out += images[:, yi.clip(0, h - 1), xi.clip(0, w - 1)] * (weight * inside)
    return out


def grid_scores(maps):
    """Sixfold symmetry of each map: min(r60, r120) - max(r30, r90, r150), best over ring sizes."""
    sac = autocorrelogram(maps)
    n, size, _ = sac.shape
    res = maps.shape[1]
    r = np.hypot(*(np.mgrid[0:size, 0:size] - (size - 1) / 2))
    rotated = {a: rotate(sac, a).reshape(n, -1) for a in (30, 60, 90, 120, 150)}
    flat = sac.reshape(n, -1)
    best = np.full(n, -np.inf)
    # Rings stop at ring_max of the map width. Beyond it two copies of the map
    # barely overlap, the autocorrelogram is noise, and in the first T4 run that
    # noise gave single blobs scores above 1.1. The cap keeps full sensitivity to
    # lattices up to 0.7 of the box apart.
    for outer in np.linspace(0.4, cfg["ring_max"], 10):
        ring = ((r >= 0.2 * res) & (r <= outer * res)).reshape(-1)
        a = flat[:, ring] - flat[:, ring].mean(1, keepdims=True)
        corr = {}
        for angle, rot in rotated.items():
            b = rot[:, ring] - rot[:, ring].mean(1, keepdims=True)
            corr[angle] = (a * b).sum(1) / np.sqrt((a**2).sum(1) * (b**2).sum(1) + 1e-12)
        score = np.minimum(corr[60], corr[120]) - np.maximum(
            np.maximum(corr[30], corr[90]), corr[150]
        )
        best = np.maximum(best, score)
    return best, sac


def show_maps(maps, scores, count, cmap):
    order = np.argsort(-scores)[:count]  # callers pass unstable units as -inf
    cols = math.ceil(math.sqrt(count))
    rows = math.ceil(count / cols)
    _, axes = plt.subplots(rows, cols, figsize=(1.8 * cols, 1.95 * rows))
    for ax, unit in zip(axes.ravel()[: len(order)], order, strict=True):
        ax.imshow(maps[unit].T, origin="lower", cmap=cmap, interpolation="gaussian")
        ax.set_title(f"{scores[unit]:.2f}", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return order


def grid_units(maps, scores, stability):
    """A grid unit scores above the cut AND draws the same map from both halves of the walks.

    Returns (mask over units, percent of active units)."""
    active = maps.std(axis=(1, 2)) > 1e-6  # silent units have no map to score
    mask = active & (scores > cfg["grid_score_cut"]) & (stability > cfg["stability_min"])
    return mask, round(100 * float(mask[active].mean()), 1)


# The null. In the same network before training, maps are speckle, and speckle
# can score above 1 by pure chance: on T4 runs about 15% of untrained units
# cleared a score of 0.3, as many as in the trained network. Speckle cannot
# repeat itself across two halves of the data, so the stability test removes it.
torch.manual_seed(cfg["seed"])
untrained = PathIntegrator(cfg["place_cells"], cfg["hidden_units"]).to(device).eval()
untrained_maps, untrained_stability, *_ = survey(untrained, cfg["surround_scale"], cfg["seq_len"])
_, grid_percent_untrained = grid_units(
    untrained_maps, grid_scores(untrained_maps)[0], untrained_stability
)

dog_scores, dog_sac = grid_scores(dog_maps)
control_scores, _ = grid_scores(control_maps)
dog_grid, grid_percent_dog = grid_units(dog_maps, dog_scores, dog_stability)
control_grid, grid_percent_control = grid_units(control_maps, control_scores, control_stability)
active_dog = dog_maps.std(axis=(1, 2)) > 1e-6
active_control = control_maps.std(axis=(1, 2)) > 1e-6
grid_advantage_points = round(grid_percent_dog - grid_percent_control, 1)
cut = cfg["grid_score_cut"]
best_grid_score = round(float(dog_scores.max()), 2)

stable_dog = np.where(dog_stability > cfg["stability_min"], dog_scores, -np.inf)
top_dog_units = show_maps(dog_maps, stable_dog, cfg["units_shown"], "inferno")

if env.lang == "ar":
    print(
        f"وحدات سداسية مستقرة · {grid_percent_dog}% بعد التدريب · {grid_percent_untrained}% بالأوزان نفسها قبله"
    )
else:
    print(
        f"stable grid units · trained {grid_percent_dog}% · same weights before training {grid_percent_untrained}%"
    )

_First the twin's best stable units, laid out the same way. A few can score well while showing a single blob: the score is not a perfect detector, which is why the percentages carry the result and no single map does. Then the distribution of grid scores over all active units, orange for centre-surround, blue for Gaussian, the dashed line at the cut, and beside it the autocorrelogram of one steady grid unit. In a clean lattice the peaks nearest the centre form a hexagon; at this resolution expect that pattern to show through noise, not crisply._

In [ ]:
stable_control = np.where(control_stability > cfg["stability_min"], control_scores, -np.inf)
top_control_units = show_maps(control_maps, stable_control, cfg["units_shown"], "viridis")

fig, (ax_hist, ax_sac) = plt.subplots(1, 2, figsize=(9, 3.8))
bins = np.linspace(-1.0, 1.8, 57)
ax_hist.hist(control_scores[active_control], bins=bins, color="tab:blue", alpha=0.55, density=True)
ax_hist.hist(dog_scores[active_dog], bins=bins, color="tab:orange", alpha=0.55, density=True)
ax_hist.axvline(cut, color="0.2", ls="--", lw=1)
# The exemplar is the steadiest grid unit that fires over a real part of the
# floor. The top scorer can be a unit with a few tiny spots, which scores well
# and draws a noisy autocorrelogram; a stable, well-covered map draws a cleaner one.
coverage = (dog_maps > 0.5 * dog_maps.max(axis=(1, 2), keepdims=True)).mean(axis=(1, 2))
candidates = np.flatnonzero(dog_grid & (coverage >= 0.15))
exemplar = (
    int(candidates[np.argmax(dog_stability[candidates])])
    if len(candidates)
    else int(top_dog_units[0])
)
ax_sac.imshow(dog_sac[exemplar].T, origin="lower", cmap="RdBu_r", vmin=-1, vmax=1)
ax_sac.axis("off")
plt.tight_layout()
plt.show()

if env.lang == "ar":
    print(
        f"وحدات سداسية مستقرة · هدف المركز والمحيط {grid_percent_dog}% · الهدف الغاوسي {grid_percent_control}%"
    )
else:
    print(
        f"stable grid units · centre-surround {grid_percent_dog}% · Gaussian {grid_percent_control}%"
    )

In this run {{run.metrics.grid_percent_dog}}% of the centre-surround network's active units were stable grid units, against {{run.metrics.grid_percent_control}}% for its twin and {{run.metrics.grid_percent_untrained}}% for the same weights before training. The twin was also the better navigator. The network that grew hexagons is not the one that solved the task best, so hexagons are not what path integration needs here.

Sorscher, Mel, Ganguli and Ocko proposed the mechanism in 2019. A surround penalises both very broad and very fine spatial patterns, so the target itself favours units built around one preferred spatial frequency. Firing rates cannot go negative, and among patterns built from a single frequency under that constraint, the triangular lattice wins. Remove the surround and the preferred frequency goes, taking the lattice with it. The pull comes from the shape of the target, and in this run it held even though it bought no better navigation.

Schaeffer, Khona and Fiete pressed the point in 2022: emergence of this kind depends on choices a modeller makes, and a network reproducing a brain's pattern is evidence about the objective, not proof that the brain is optimising it. This run is a small instance of their argument.

One caution before leaning on the navigation gap. Position is decoded from the three most active predicted place cells, and about three quarters of the centre-surround code is a nearly flat floor, so its peak is harder to read. Part of the gap may come from a harder readout rather than worse integration, and this run cannot separate the two.

The hexagons were never asked for. The shape of the target asked for them, one step removed, and they arrived without making the network any more accurate.

> **The paper** · [recurrent-neural-network](https://azimuth.plus/en/paper/recurrent-neural-network) — a hidden state that carries the position forward from step to step
>
> The entire map of the room is stored in one vector updated by the same weights at every step.

> **The paper** · [backpropagation-through-time](https://azimuth.plus/en/paper/backpropagation-through-time) — how an error at the last step reaches the weights used at the first
>
> Every walk is unrolled over all of its steps; the lattice is shaped by gradients that cross every one of them.

### Exercise — longer-walks

The network never saw a walk longer than {{scale.seq_len}} steps. Raise LENGTH_MULTIPLE and score only the steps beyond that horizon. The top row redraws the best units from before; the bottom row is the same units on long walks. The printout compares position error in centimetres inside and beyond the horizon, next to the error of always guessing the box centre. Decide which breaks first, the position readout or the lattice, and what that says about where the network keeps its sense of place.

_A hint is available: `env.hint(1)`_

In [ ]:
# YOUR TURN.
# The network only ever saw walks of cfg["seq_len"] steps. Run it for longer and
# score only the steps it was never trained on. Try 1, then 3, 5, 10.
# Errors are compared in centimetres: skill is measured against standing still,
# and standing still gets worse on longer walks, which would flatter the network.
LENGTH_MULTIPLE = 5

long_len = cfg["seq_len"] * LENGTH_MULTIPLE
long_maps, _, _, long_err_cm, centre_cm = survey(
    dog_model, cfg["surround_scale"], long_len, skip=cfg["seq_len"] if LENGTH_MULTIPLE > 1 else 0
)
long_scores, _ = grid_scores(long_maps)
kept = [float(long_scores[u]) for u in top_dog_units]

fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for i, unit in enumerate(top_dog_units[:8]):
    axes[0, i].imshow(dog_maps[unit].T, origin="lower", cmap="inferno", interpolation="gaussian")
    axes[1, i].imshow(long_maps[unit].T, origin="lower", cmap="inferno", interpolation="gaussian")
    axes[0, i].set_title(f"{dog_scores[unit]:.2f}", fontsize=9)
    axes[1, i].set_title(f"{long_scores[unit]:.2f}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

if env.lang == "ar":
    print(
        f"مسارات أطول بـ {LENGTH_MULTIPLE} مرات · خطأ الموضع بعد أفق التدريب {long_err_cm:.1f} سم"
    )
    print(f"داخل الأفق {dog_err_cm:.1f} سم · تخمين مركز الصندوق دائماً {centre_cm:.1f} سم")
    print(
        f"متوسط درجة الوحدات نفسها {np.mean(kept):.2f} (كان {np.mean(dog_scores[top_dog_units]):.2f})"
    )
else:
    print(
        f"walks {LENGTH_MULTIPLE}× longer · position error beyond the training horizon {long_err_cm:.1f} cm"
    )
    print(
        f"within the horizon {dog_err_cm:.1f} cm · always guessing the box centre {centre_cm:.1f} cm"
    )
    print(
        f"mean score of the same units {np.mean(kept):.2f} (was {np.mean(dog_scores[top_dog_units]):.2f})"
    )

> **The paper** · [long-short-term-memory](https://azimuth.plus/en/paper/long-short-term-memory) — the gated alternative this workshop deliberately does not use
>
> DeepMind's 2018 grid-cell agent used an LSTM; here a plain ReLU recurrence is enough, which makes gating an option, not a precondition.

> **The paper** · [neural-turing-machine](https://azimuth.plus/en/paper/neural-turing-machine) — a contrast: memory with an address, against memory that is only a state
>
> A Neural Turing Machine writes to a location it can look up. This network has no address to write to — the lattice is how it indexes space instead.

_Navigation first; the grid comparison is only meaningful if both networks passed it._

In [ ]:
# Control first: a grid comparison between networks that cannot find their way
# would be a comparison between two kinds of noise. A grid unit must both score
# above the cut and reproduce its map from independent halves of the walks.
integrates_ok = env.check("both-integrate", min(dog_skill, control_skill))
grids_ok = env.check("grid-units", grid_percent_dog)
advantage_ok = env.check("surround-advantage", grid_advantage_points)

_Your completion code and the provenance of this run._

In [ ]:
receipt = env.receipt()